# Advanced HEN Methods

**Learning outcome:** Apply advanced hen methods through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Advanced  
**Execution profile:** `solver`  
**Expected runtime:** 10 to 90 minutes  
**Optional extras:** `synthesis`

## Study question and data

**Study question:** How do alternative synthesis and network-improvement methods change the candidate design?

## Step 1: Run a deeper synthesis

This cell loads the four-stream Yee and Grossmann case and runs `heat_exchanger_network(depth=3)`. Search depth 3 adds the MILP pinch design and thermal derivative methods to the default depth 1, makes more pinch design networks and evolves them more widely; `width` would widen the dt_cont multipliers, which default to 0.75, 1.0 and 1.5. `deeper` holds the result, and `deeper_ranking` is its best successful outcome.

In [ ]:
from OpenPinch import PinchProblem

problem = PinchProblem(
    "Four-stream-Yee-and-Grossmann-1990-1.json", project_name="Four Stream"
)
deeper = problem.design.heat_exchanger_network(depth=3)
deeper_ranking = deeper.top(1)

## Step 2: Compare advanced design methods

This cell runs the design methods one at a time. `pinch_design()` runs the pinch design step alone: the rules-based (Linnhoff) method, which proposes networks from the CP and number-of-streams rules at the pinch, at the default depth 1. From depth 2 the MILP method runs too. `thermal_derivative()` is seeded with the pinch design network, and `network_evolution()` with the thermal-derivative network. Compare the `*_ranking` outputs.

In [ ]:
pinch_design = problem.design.pinch_design()
pinch_ranking = pinch_design.top(1)
with_milp = problem.design.pinch_design(options={"HENS_SEARCH_DEPTH": 2})
with_milp_ranking = with_milp.top(1)
thermal = problem.design.thermal_derivative((pinch_design.selected_network,))
thermal_ranking = thermal.top(1)
evolution = problem.design.network_evolution((thermal.selected_network,))
evolution_ranking = evolution.top(1)

## Step 3: Optimise duties on a hand-built network

This cell defines a network structure by hand: four recovery matches in two stages as `(hot stream, cold stream, stage)`, a heater on Milk Concentrate and a cooler on each hot stream. `optimise_duties()` keeps that structure and allocates duty (and stream splits) three ways. `min_utility` minimises utility use with at least 10 K approach on every exchanger. `min_area` minimises total area with hot utility capped at 1.5 times that minimum. `min_cost` minimises total annual cost, requiring only a 1 K approach. Each result is a normal design view, so `top()`, `grid()` and the utility totals work as for the synthesis methods.

In [ ]:
structure = {
    "recovery": [
        ("Raw Milk", "Milk Concentrate", 1),
        ("HT Flash", "Milk Concentrate", 1),
        ("Raw Milk", "CIP Water", 2),
        ("HT Flash", "CIP Water", 2),
    ],
    "heaters": ["Milk Concentrate"],
    "coolers": ["Raw Milk", "HT Flash"],
}
min_utility = problem.design.optimise_duties(
    structure, objective="utility", min_approach_temperature=10.0
)
min_area = problem.design.optimise_duties(
    structure,
    objective="area",
    min_approach_temperature=10.0,
    max_hot_utility=1.5 * min_utility.total_hot_utility,
)
min_cost = problem.design.optimise_duties(structure, objective="cost")
duty_rankings = [view.top(1) for view in (min_utility, min_area, min_cost)]

## Step 4: Retrofit an existing network

This cell loads the five-stream Ciric and Floudas retrofit case and describes the network as built: four recovery exchangers in two stages, a heater on C2 and coolers on H1 and H2, each with its installed area (m2) and plant tag. `retrofit()` rates that network at its installed areas, then makes one structural change at a time, up to `max_modifications`: here new matches, ranked by the utility their structure could reach with unlimited area, each solved with the added area its existing exchangers need. Existing area is sunk; added area costs the `COSTING_HX_AREA_*` law, a new exchanger `COSTING_HX_UNIT_COST`, and here an area addition $8,650 and a stream split $10,000. `retrofit.frontier` holds the best design for each number of structural changes, from the existing network (0) up; `retrofit_frontier` lists each one's hot utility (kW), net annual benefit ($/y), payback (years) and modifications, area additions included.

In [ ]:
retrofit_problem = PinchProblem(
    "Five-stream-Ciric-and-Floudas-1989.json", project_name="Five Stream Retrofit"
)
existing = {
    "recovery": [
        {"hot": "H3", "cold": "C2", "stage": 1, "area": 609.7, "id": "E1"},
        {"hot": "H3", "cold": "C1", "stage": 2, "area": 579.2, "id": "E2"},
        {"hot": "H1", "cold": "C1", "stage": 1, "area": 1008.5, "id": "E3"},
        {"hot": "H2", "cold": "C2", "stage": 2, "area": 117.96, "id": "E4"},
    ],
    "heaters": [{"stream": "C2", "area": 246.75, "id": "E7"}],
    "coolers": [
        {"stream": "H1", "area": 787.5, "id": "E5"},
        {"stream": "H2", "area": 104.6, "id": "E6"},
    ],
}
retrofit = retrofit_problem.design.retrofit(
    existing,
    max_modifications=3,
    area_addition_cost=8_650.0,
    split_cost=10_000.0,
)
retrofit_frontier = [
    {
        "structural_changes": sum(
            m.kind != "area_addition" for m in design.modifications
        ),
        "hot_utility_kW": round(design.hot_utility),
        "net_annual_benefit": round(design.net_annual_benefit),
        "payback_years": design.payback_years,
        "steps": [(m.kind, m.exchanger_id) for m in design.modifications],
    }
    for design in retrofit.frontier
]

## Review the result

Compare the leading result from every method on the same stream and economic basis before attributing design differences to the search method.

In [ ]:
from IPython.display import display

display(deeper_ranking)
display(pinch_ranking)
display(with_milp_ranking)
display(thermal_ranking)
display(evolution_ranking)
display(duty_rankings)
display(retrofit_frontier)

## Interpret the result

Compare every method on the same streams, utility assumptions, and economic basis; record solver status with each result.

## Adapt this template

Start with one method and small search bounds, then pass only reviewed networks into derivative or evolution workflows.